# Incremental/streaming Recommendation on basis of scenario

| #  | Source Type       | Load Type                       | Trigger / Timing    | Incremental? | Spark / Databricks Approach        | Recommendation          | Why                               |
| -- | ----------------- | ------------------------------- | ------------------- | ------------ | ---------------------------------- | ----------------------- | --------------------------------- |
| 1  | Files (ADLS / S3) | Full Load                       | One-time / Initial  | ❌ No         | `spark.read`                       | ⭐ Use normal batch read | Simple, no state needed           |
| 2  | Files (ADLS / S3) | Incremental                     | Daily at fixed time | ✅ Yes        | **Auto Loader + trigger once**     | ⭐⭐⭐ Best                | Tracks processed files            |
| 3  | Files (ADLS / S3) | Incremental + Complex Transform | Daily at fixed time | ✅ Yes        | **Auto Loader + Spark transforms** | ⭐⭐⭐ Best                | Scalable + transformation support |
| 4  | Files (ADLS / S3) | Incremental                     | On file arrival     | ✅ Yes        | **Auto Loader (continuous)**       | ⭐⭐⭐ Best                | Event-driven ingestion            |
| 5  | Files (ADLS / S3) | Incremental                     | Daily at fixed time | ❌ (manual)   | `spark.read` + filter              | ❌ Not recommended       | Full scan every run               |
| 6  | Delta Table       | Full Load                       | Initial / Rebuild   | ❌ No         | `spark.read.format("delta")`       | ⭐ Use batch read        | Simple overwrite                  |
| 7  | Delta Table       | Incremental (Insert only)       | Daily at fixed time | ✅ Yes        | **Delta Change Data Feed (CDF)**   | ⭐⭐⭐ Best                | Reads only new rows               |
| 8  | Delta Table       | Incremental (Insert + Update)   | Daily at fixed time | ✅ Yes        | **CDF + MERGE**                    | ⭐⭐⭐ Best                | Handles updates correctly         |
| 9  | Delta Table       | Incremental (Insert only)       | Near real-time      | ✅ Yes        | **Streaming + CDF**                | ⭐⭐⭐ Best                | Low latency                       |
| 10 | Delta Table       | Incremental (Insert + Update)   | Near real-time      | ✅ Yes        | **Streaming CDF + MERGE**          | ⭐⭐⭐ Best                | Exactly-once processing           |


## 1) Daily Files → Insert at Fixed Time (No Complex Logic)
### Recommended: Auto Loader with trigger(once=True)
Why:
- Incremental file tracking
- Exactly-once
- No full folder scan every day

In [0]:
df = (
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/schema/daily")
    .load("/raw/daily")
)

df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk/daily") \
  .trigger(once=True) \
  .table("bronze_daily")


## 2) Daily Files → Complex Transformations → Delta
### Recommended: Auto Loader + Spark Transformations
Why:
- Same incremental ingestion
- Transformations run inside micro-batch

**Auto Loader ≠ no transformations**
It’s just the ingestion layer.

In [0]:
from pyspark.sql.functions import *

transformed_df = (
  df
  .withColumn("ingest_date", current_date())
  .withColumn("amount", col("amount").cast("decimal(10,2)"))
  .filter(col("status") == "ACTIVE")
)

## 3) Incremental Source Delta → Only Inserts (No Updates)
### Recommended: Delta Change Data Feed (CDF)

Why
- No need to scan full table
- Reads only new rows since last version

In [0]:
%sql
-- enable CDC
ALTER TABLE source_table
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
df = (
    spark.read.format('delta')\
        .option('readChangefEED', 'true')\
            .option('startingVersion',last_version)\
                .table('source_table_name')\
                    .filter("_change_type = 'insert'")
)

df.write.mode("append").saveAsTable("target_table")

## 4) Incremental Source Delta → Inserts + Updates
### Recommended: CDF + MERGE

Why
- Handles updates & inserts
- Maintains correctness

In [0]:
df = spark.read.format("delta")\
    .option("readChangeFeed","true")\
        .option("startingTimestamp", last_ts)\
            .table("source_table_name")

df.createOrReplaceTempView("source_table_changes")

In [0]:
%sql
-- Merge and insert using Pyspark(Python)
from delta.tables import DeltaTable
target_dt = DeltaTable.forName(spark, "target_table_name")

target_dt.alias("t")\
        .merge(
          source_table_name.alias("s"),
          condition="t.id = s.id"
        )\
        .whenMatchedUpdate(
          condition = "s._change_type = 'update_postimage'",
          set={col: f"s.{col}" for col in source_table_name.columns}
        )\
        .whenNotMatchedInsert(
        condition="s._change_type = 'insert'",
        values={col: f"s.{col}" for col in changes.columns}
        )\
        .execute()

In [0]:
%sql
-- Merge and insert using SQL
MERGE INTO target_table as t
USING source_table_changes as s
on t.id = s.id
WHEN MATCHED AND s._change_type = "update_postimage" THEN 
  UPDATE SET *
WHEN NOT MATCHED AND s._change_type = "insert" THEN
  INSERT *

## 5) Daily Files → Insert Based on File Arrival
### Recommended: Auto Loader (Continuous Mode)

Why
- Processes files immediately
- Event-driven
- Best for near real-time

No scheduling needed, 
Runs continuously

In [0]:
#we can set event based triggers also
df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/chk/arrival") \
  .trigger(processingTime="5 minutes") \  
  .table("bronze_arrival")

## 6) Streaming from a Delta Table (Real-Time / Near Real-Time)
- Streaming mode automatically handles incremental changes.
- checkpointLocation is mandatory for exactly-once guarantees.
- Use outputMode("append") or update depending on target.

In [0]:
# continuous stream from source Delta table
df_stream = spark.readStream.format('delta')
.option('readChangeFeed','true')
.option('source_delta_table')

# filter insert or update
df_stream_filter = df_stream.filter("_change_type in ('update','insert')")

# Optional trasformation
df_stream_transformed = df_stream_filtered.withColumn('ingest_ts', current_timestamp())

# Write to target table 
query = df_stream_transformed.writeStream.format("delta")\
                                        .option("checkpointLocation", "/chk/arrival") \
                                        .outputMode("append") \
                                        .table("bronze_arrival")
query.awaitTermination()